In [0]:
%run "/Workspace/DAV Project/YouTube-Trending-Analytics/src/pipelines/silver_to_gold"

In [0]:
import sys, os
sys.path.append(os.path.abspath(".."))
from src.params import resolve_params
from src.pipelines.silver_to_gold import run_silver_to_gold, validate_gold

In [0]:
#Check the source
from pyspark.sql import functions as F

SILVER = "workspace.youtube_trending.silver_video_daily"

silver = spark.table(SILVER)

print("Silver rows:", silver.count())

silver.select(
    F.min("snapshot_date").alias("earliest_date"),
    F.max("snapshot_date").alias("latest_date"),
    F.countDistinct("snapshot_date").alias("snapshot_dates")
).show()

Silver rows: 10289
+-------------+-----------+--------------+
|earliest_date|latest_date|snapshot_dates|
+-------------+-----------+--------------+
|   2026-09-26| 2026-11-14|            20|
+-------------+-----------+--------------+



In [0]:
dbutils.widgets.dropdown("load_type", "incremental", ["full", "incremental", "backfill"])
dbutils.widgets.text("start_date", "")
dbutils.widgets.text("end_date", "")
dbutils.widgets.text("batch_id", "")

load_type = dbutils.widgets.get("load_type")
start, end, _, batch_id = resolve_params(
    load_type, dbutils.widgets.get("start_date"),
    dbutils.widgets.get("end_date"), "", dbutils.widgets.get("batch_id"))

from pyspark.sql import functions as F

# Existing Gold tables were created with an older schema (gold_dim_date still
# has month_number/day_of_month, the fact has views/likes), so every MERGE fails
# with DELTA_MERGE_UNRESOLVED_EXPRESSION. One-time fix: reset the Gold tables
# and rebuild the full silver range so no existing Gold data is lost.
old_schema = "month_number" in spark.table("workspace.youtube_trending.gold_dim_date").columns
if old_schema:
    rng = spark.table("workspace.youtube_trending.silver_video_daily").agg(
        F.min("snapshot_date").alias("mn"), F.max("snapshot_date").alias("mx")).first()
    start, end = str(rng["mn"]), str(rng["mx"])
    load_type = "full"

result = run_silver_to_gold(spark, start, end, load_type, batch_id, reset=old_schema)
print(result)
print(validate_gold(spark))

{'batch_id': 'incremental_2026-11-14_2026-11-14_bbe00ac6', 'silver_rows': 10289, 'fact_rows': 10289, 'tables': {'workspace.youtube_trending.gold_dim_date': {'read': 20, 'inserted': 20, 'updated': 0}, 'workspace.youtube_trending.gold_dim_region': {'read': 10, 'inserted': 10, 'updated': 0}, 'workspace.youtube_trending.gold_dim_category': {'read': 33, 'inserted': 33, 'updated': 0}, 'workspace.youtube_trending.gold_dim_channel': {'read': 2900, 'inserted': 2900, 'updated': 0}, 'workspace.youtube_trending.gold_dim_video': {'read': 5383, 'inserted': 5383, 'updated': 0}, 'workspace.youtube_trending.gold_fact_video_daily': {'read': 10289, 'inserted': 10289, 'updated': 0}}}
{'row_counts': {'workspace.youtube_trending.gold_dim_date': 20, 'workspace.youtube_trending.gold_dim_region': 10, 'workspace.youtube_trending.gold_dim_category': 33, 'workspace.youtube_trending.gold_dim_channel': 2900, 'workspace.youtube_trending.gold_dim_video': 5383, 'workspace.youtube_trending.gold_fact_video_daily': 10289

In [0]:
import uuid

gold_result = run_silver_to_gold(
    spark=spark,
    start_date="2026-09-26",
    end_date="2026-10-25",
    load_type="full",
    batch_id=f"gold_{uuid.uuid4().hex[:8]}"
)

print(gold_result)

{'batch_id': 'gold_80df26a2', 'silver_rows': 7059, 'fact_rows': 10289, 'tables': {'workspace.youtube_trending.gold_dim_date': {'read': 14, 'inserted': 0, 'updated': 0}, 'workspace.youtube_trending.gold_dim_region': {'read': 10, 'inserted': 0, 'updated': 0}, 'workspace.youtube_trending.gold_dim_category': {'read': 33, 'inserted': 0, 'updated': 0}, 'workspace.youtube_trending.gold_dim_channel': {'read': 2214, 'inserted': 0, 'updated': 0}, 'workspace.youtube_trending.gold_dim_video': {'read': 3698, 'inserted': 0, 'updated': 0}, 'workspace.youtube_trending.gold_fact_video_daily': {'read': 7059, 'inserted': 0, 'updated': 0}}}


In [0]:
#Check for duplicate fact records
from pyspark.sql import functions as F

fact = spark.table("workspace.youtube_trending.gold_fact_video_daily")

duplicates = (
    fact.groupBy("video_id", "region_code", "snapshot_date")
    .count()
    .filter(F.col("count") > 1)
)

print("Duplicate fact keys:", duplicates.count())

Duplicate fact keys: 0


In [0]:
from pyspark.sql import functions as F

# 1. Check row counts
tables = [
    "gold_dim_date",
    "gold_dim_region",
    "gold_dim_category",
    "gold_dim_channel",
    "gold_dim_video",
    "gold_fact_video_daily"
]

for name in tables:
    df = spark.table(f"workspace.youtube_trending.{name}")
    print(f"{name}: {df.count():,} rows")

# 2. Check for null fact keys
fact = spark.table(
    "workspace.youtube_trending.gold_fact_video_daily"
)

fact.select(
    F.sum(F.col("video_id").isNull().cast("int")).alias("null_video_id"),
    F.sum(F.col("region_code").isNull().cast("int")).alias("null_region"),
    F.sum(F.col("snapshot_date").isNull().cast("int")).alias("null_snapshot_date"),
    F.sum(F.col("date_key").isNull().cast("int")).alias("null_date_key")
).show()

# 3. Check date dimension key uniqueness
date_dim = spark.table(
    "workspace.youtube_trending.gold_dim_date"
)

print(
    "Duplicate date keys:",
    date_dim.groupBy("date_key").count()
        .filter(F.col("count") > 1).count()
)

# 4. Preview fact table
fact.orderBy(
    F.col("snapshot_date").desc()
).show(10, truncate=False)

gold_dim_date: 20 rows
gold_dim_region: 10 rows
gold_dim_category: 33 rows
gold_dim_channel: 2,900 rows
gold_dim_video: 5,383 rows
gold_fact_video_daily: 10,289 rows
+-------------+-----------+------------------+-------------+
|null_video_id|null_region|null_snapshot_date|null_date_key|
+-------------+-----------+------------------+-------------+
|            0|          0|                 0|            0|
+-------------+-----------+------------------+-------------+

Duplicate date keys: 0
+--------+-----------+-----------+-------------+-----------+----------------------------------------------------------------+----------+----------+-------------+-------------+---------------------+-------------+------------------------------------------+----------------------------------------------------------------+--------------------------+
|date_key|video_id   |region_code|snapshot_date|category_id|channel_key                                                     |view_count|like_count|dislike_cou

In [0]:
dbutils.widgets.dropdown("load_type", "incremental", ["full", "incremental", "backfill"])
dbutils.widgets.text("start_date", "")
dbutils.widgets.text("end_date", "")
dbutils.widgets.text("batch_id", "")

load_type = dbutils.widgets.get("load_type")
start, end, _, batch_id = resolve_params(
    load_type, dbutils.widgets.get("start_date"),
    dbutils.widgets.get("end_date"), "", dbutils.widgets.get("batch_id"))

result = run_silver_to_gold(spark, start, end, load_type, batch_id)
print(result)
print(validate_gold(spark))

{'batch_id': 'incremental_2026-11-14_2026-11-14_15d4c85f', 'silver_rows': 738, 'fact_rows': 10289, 'tables': {'workspace.youtube_trending.gold_dim_date': {'read': 1, 'inserted': 0, 'updated': 0}, 'workspace.youtube_trending.gold_dim_region': {'read': 10, 'inserted': 0, 'updated': 0}, 'workspace.youtube_trending.gold_dim_category': {'read': 33, 'inserted': 0, 'updated': 0}, 'workspace.youtube_trending.gold_dim_channel': {'read': 492, 'inserted': 0, 'updated': 0}, 'workspace.youtube_trending.gold_dim_video': {'read': 531, 'inserted': 0, 'updated': 0}, 'workspace.youtube_trending.gold_fact_video_daily': {'read': 738, 'inserted': 0, 'updated': 0}}}
{'row_counts': {'workspace.youtube_trending.gold_dim_date': 20, 'workspace.youtube_trending.gold_dim_region': 10, 'workspace.youtube_trending.gold_dim_category': 33, 'workspace.youtube_trending.gold_dim_channel': 2900, 'workspace.youtube_trending.gold_dim_video': 5383, 'workspace.youtube_trending.gold_fact_video_daily': 10289}, 'duplicate_fact_k